# Finetune the BERT model

[Hugging Face Tutorial](https://huggingface.co/docs/transformers/v4.16.2/en/training)

- **Dataset:** [IMDB](https://data.imdb.com/non-commercial-datasets/)
- **Task:** Classify whether movie reviews are positive or negative

NOTEBOOK: [Next Sentence Prediction with BERT](https://github.com/ufdatastudio/predictions/blob/development/notebook_experiments/tutorial-next_sentence_prediction-bert.ipynb)

In [1]:
import os
import sys

import numpy as np

from datasets import load_dataset

from transformers import Trainer
from transformers import AutoTokenizer
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification

from sklearn.metrics import accuracy_score

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

# import log_files
from data_processing import DataProcessing

In [2]:
hf_api_key = os.getenv("HUGGING_FACE_TEXT_CLASSIFICATION_API_KEY")

## Load Dataset

In [3]:
# Get original dataset
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
train_p_o_path = os.path.join(base_data_path, 'text_classification_tutorial/train_dict-v2.json')
test_p_o_path = os.path.join(base_data_path, 'text_classification_tutorial/test_dict-v2.json')
files_by_split = {
    'train': str(train_p_o_path),
    'test': str(test_p_o_path)
}
raw_datasets = load_dataset('json', data_files=files_by_split)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__'],
        num_rows: 4666
    })
    test: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__'],
        num_rows: 1167
    })
})

In [4]:
raw_datasets["test"][0]

{'Base Sentence': 'According to environmental scientist Dr. Sofia Rodriguez, the air quality index at Los Angeles would fall in August 2027.',
 'Ground Truth': 1,
 'label': 1,
 '__index_level_0__': 3550}

In [8]:
for i, x in enumerate(raw_datasets["train"]["Base Sentence"]):
    if not isinstance(x, str):
        print(i, type(x), x)
        break

305 <class 'NoneType'> None


In [9]:
raw_datasets = raw_datasets.filter(
    lambda x: x["Base Sentence"] is not None
)

Filter:   0%|          | 0/4666 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1167 [00:00<?, ? examples/s]

In [10]:
for i, x in enumerate(raw_datasets["train"]["Base Sentence"]):
    if not isinstance(x, str):
        print(i, type(x), x)
        break

In [12]:
print(raw_datasets["train"].features)
print(type(raw_datasets["train"][0]["Base Sentence"]))

{'Base Sentence': Value('string'), 'Ground Truth': Value('int64'), 'label': Value('int64'), '__index_level_0__': Value('int64')}
<class 'str'>


## Preprocess: Tokenize + Datasplits

In [13]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

### 1. Tokenize

Apply these preprocessing steps to all the splits of our dataset

In [14]:
def tokenize_function(examples):
    return tokenizer(examples["Base Sentence"], padding="max_length", truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/4647 [00:00<?, ? examples/s]

Map:   0%|          | 0/1162 [00:00<?, ? examples/s]

In [15]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 4647
    })
    test: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 1162
    })
})

### 2. Datasplits

Generate a small subset of the training and validation set, to enable faster training:

In [16]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))
full_train_dataset = tokenized_datasets["train"]
full_eval_dataset = tokenized_datasets["test"]

## Finetune

Since PyTorch does not provide a training loop, the 🤗 Transformers library provides a Trainer API that is optimized for 🤗 Transformers models, with a wide range of training options and with built-in features like logging, gradient accumulation, and mixed precision.

### Define Model

- We will fine-tune this model on our task, transferring the knowledge of the pretrained model to it (which is why doing this is called transfer learning).

In [17]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

#### Model Details + Learnings

1. BOOK: [Dive Into DL](https://d2l.ai/chapter_natural-language-processing-pretraining/bert.html)

BERT = Bidrectional Encoder Representations from Transformers

![alt text](<Screenshot 2026-09-17 at 10.51.15.png>)

- Unidirectional vs Bidirectional
- Multidirectional (Bi + Hop or Segment (like n-grams)) - DJB

### Define our Trainer

In [18]:
training_args = TrainingArguments("test_trainer")
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,


### Initiate a Trainer

In [19]:
trainer = Trainer(model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset)
trainer

### Finetune model

In [20]:
trainer.train()

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=375, training_loss=0.08085698954264323, metrics={'train_runtime': 133.4669, 'train_samples_per_second': 22.477, 'train_steps_per_second': 2.81, 'total_flos': 789333166080000.0, 'train_loss': 0.08085698954264323, 'epoch': 3.0})

### Metrics

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)
trainer.evaluate(),

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


({'eval_loss': 0.05920041725039482,
  'eval_model_preparation_time': 0.001,
  'eval_accuracy': 0.991,
  'eval_runtime': 13.1523,
  'eval_samples_per_second': 76.032,
  'eval_steps_per_second': 9.504},)